# 02 · Exploratory data analysis and subset selection

### What this notebook does
1. Profiles the raw 5-core interactions: activity per user and item, ratings, time span, sparsity, long tail.
2. Checks data quality: missing values, duplicate user–item pairs, timestamp ties, metadata coverage.
3. Profiles product metadata: category tree, how many items have a title, description, features, brand.
4. **Compares candidate subsets** (level-1 sub-categories and random user samples), re-applying 5-core to each.
5. **Recommends one subset** using explicit, written-down rules and saves the choice to `interim/subset_choice.json`.

### Why
- The full 5-core set (≈730K users, ≈208K items) is too large for affordable LLM distillation (Model 1) and for
  full-catalog ranking. We need a subset that is small enough to train but large enough for federated (Model 2)
  and DP (Model 4) experiments, which both need many users.
- Every cleaning and filtering decision in notebooks 03–05 should be justified by a number from this notebook.

**Inputs:** `raw/ratings_5core.parquet`, `raw/meta_5core_items.parquet` · **Outputs:** figures in `reports/figures/`,
`reports/eda_summary.json`, `reports/subset_comparison.csv`, `interim/subset_choice.json`

In [ ]:
# ---------------- Shared setup (identical in every notebook) ----------------
# All notebooks read and write inside one project folder (BASE).
# Colab: BASE lives in your Google Drive so outputs survive runtime restarts.
# Local Jupyter: BASE is ./298A_recsys next to the notebooks.
# Set the environment variable RECSYS_BASE to use any other folder.
import os, sys, json, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules
if os.environ.get('RECSYS_BASE'):
    BASE = Path(os.environ['RECSYS_BASE'])
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/298A_recsys')
else:
    BASE = Path('./298A_recsys').resolve()

RAW, INTERIM, PROCESSED = BASE / 'raw', BASE / 'interim', BASE / 'processed'
REPORTS, FIGS = BASE / 'reports', BASE / 'reports' / 'figures'
# Large downloads go to fast local disk in Colab (not Drive); re-downloaded if the runtime resets.
DOWNLOADS = Path(os.environ.get('RECSYS_DOWNLOAD_DIR', '/content/downloads' if IN_COLAB else str(BASE / 'downloads')))
for d in [RAW, INTERIM, PROCESSED, REPORTS, FIGS, DOWNLOADS]:
    d.mkdir(parents=True, exist_ok=True)

# Plot style: one hue per single-series chart, light recessive grid, no chart junk.
PRIMARY, ACCENT, MUTED = '#2a78d6', '#eb6834', '#8a8984'
plt.rcParams.update({
    'figure.figsize': (8, 4), 'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.edgecolor': '#b5b4ae',
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.prop_cycle': matplotlib.cycler(color=[PRIMARY, ACCENT, '#1baf7a', '#eda100']),
})

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight')
    plt.show()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_config():
    cfg_path = BASE / 'config.json'
    if not cfg_path.exists():
        raise FileNotFoundError('config.json not found - run 01_data_acquisition.ipynb first.')
    return json.loads(cfg_path.read_text())

print('Project folder:', BASE)

In [ ]:
CFG = load_config()
SEED = CFG['seed']
ratings = pd.read_parquet(RAW / 'ratings_5core.parquet')
meta = pd.read_parquet(RAW / 'meta_5core_items.parquet')
ratings['dt'] = pd.to_datetime(ratings['timestamp'], unit='ms', utc=True)
print(f'{len(ratings):,} interactions | {ratings.user_id.nunique():,} users | {ratings.parent_asin.nunique():,} items')
print(f'{len(meta):,} metadata rows')

## 1. Core statistics and sparsity

In [ ]:
def core_stats(df, user_col='user_id', item_col='parent_asin'):
    u, i, n = df[user_col].nunique(), df[item_col].nunique(), len(df)
    per_user = df.groupby(user_col, observed=True).size()
    per_item = df.groupby(item_col, observed=True).size()
    return {
        'users': int(u), 'items': int(i), 'interactions': int(n),
        'density_pct': round(100 * n / max(u * i, 1), 5),
        'mean_per_user': round(per_user.mean(), 2), 'median_per_user': float(per_user.median()),
        'mean_per_item': round(per_item.mean(), 2), 'median_per_item': float(per_item.median()),
    }

stats_raw = core_stats(ratings)
pd.Series(stats_raw, name='raw 5-core').to_frame()

## 2. Activity distributions (long tail)
A long tail is expected: a few users and products account for most interactions. This motivates the
**sparse-user** and **long-tail item** analysis groups required by the report (Stage 6).

In [ ]:
per_user = ratings.groupby('user_id').size()
per_item = ratings.groupby('parent_asin').size()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, s, name in [(axes[0], per_user, 'users'), (axes[1], per_item, 'items')]:
    counts = s.value_counts().sort_index()
    ax.loglog(counts.index, counts.values, marker='o', markersize=3, linewidth=0, color=PRIMARY)
    ax.set_xlabel(f'interactions per {name[:-1]}')
    ax.set_ylabel(f'number of {name}')
    ax.set_title(f'Interactions per {name[:-1]} (log-log)')
savefig('02_activity_loglog')

print('Interactions per user:\n', per_user.describe(percentiles=[.25, .5, .75, .9, .99]).round(1))
print('\nInteractions per item:\n', per_item.describe(percentiles=[.25, .5, .75, .9, .99]).round(1))

In [ ]:
# Lorenz-style concentration: share of interactions captured by the most popular items.
sorted_pop = np.sort(per_item.values)[::-1]
cum_share = np.cumsum(sorted_pop) / sorted_pop.sum()
item_frac = np.arange(1, len(sorted_pop) + 1) / len(sorted_pop)
top20_share = cum_share[int(0.2 * len(sorted_pop)) - 1]

plt.figure(figsize=(5, 4))
plt.plot(item_frac, cum_share, color=PRIMARY, linewidth=2)
plt.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle='--')
plt.axvline(0.2, color=MUTED, linewidth=0.8)
plt.annotate(f'top 20% of items = {top20_share:.0%} of interactions', (0.21, top20_share - 0.08), fontsize=8)
plt.xlabel('fraction of items (most popular first)'); plt.ylabel('cumulative share of interactions')
plt.title('Popularity concentration')
savefig('02_popularity_concentration')

## 3. Ratings and time
Ratings are kept as side information (optional RecPO module); every review counts as an interaction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
rc = ratings.rating.value_counts().sort_index()
axes[0].bar(rc.index.astype(str), rc.values, color=PRIMARY, width=0.6)
axes[0].set_title('Rating distribution'); axes[0].set_xlabel('rating'); axes[0].set_ylabel('interactions')
monthly = ratings.set_index('dt').resample('MS').size()
axes[1].plot(monthly.index, monthly.values, color=PRIMARY, linewidth=1.5)
axes[1].set_title('Interactions per month'); axes[1].set_ylabel('interactions')
savefig('02_ratings_and_time')

print('Time span:', ratings.dt.min(), '->', ratings.dt.max())
print('Share of 5-star ratings:', f'{(ratings.rating == 5).mean():.1%}')

## 4. Data quality
- **Duplicate user–item pairs** (repeat purchases) decide the repeat policy in notebook 03.
- **Timestamp ties within a user** decide the tie-breaking rule for sequence order in notebook 05.

In [ ]:
quality = {
    'missing_values': ratings[['user_id', 'parent_asin', 'rating', 'timestamp']].isna().sum().to_dict(),
    'exact_duplicate_rows': int(ratings.duplicated(['user_id', 'parent_asin', 'rating', 'timestamp']).sum()),
    'repeat_user_item_pairs': int(ratings.duplicated(['user_id', 'parent_asin']).sum()),
    'rows_sharing_timestamp_with_another_row_of_same_user': int(ratings.duplicated(['user_id', 'timestamp'], keep=False).sum()),
    'ratings_outside_1_5': int((~ratings.rating.between(1, 5)).sum()),
    'timestamps_before_1996': int((ratings.dt < pd.Timestamp('1996-01-01', tz='UTC')).sum()),
}
quality['users_with_timestamp_ties_pct'] = round(100 * ratings[ratings.duplicated(['user_id', 'timestamp'], keep=False)].user_id.nunique() / ratings.user_id.nunique(), 2)
pd.Series(quality, dtype=object).to_frame('value')

## 5. Metadata coverage and text quality
Model 1 builds semantic item embeddings from text, so we need to know how many items have usable text.

In [ ]:
meta_u = meta.drop_duplicates('parent_asin')
items = pd.DataFrame({'parent_asin': per_item.index, 'n_inter': per_item.values})
items = items.merge(meta_u.assign(_has_meta=True), on='parent_asin', how='left')
items['_has_meta'] = items['_has_meta'].fillna(False).astype(bool)

def is_seq(v):
    return isinstance(v, (list, tuple, np.ndarray))

def nonempty_str(s):
    return s.fillna('').astype(str).str.strip().str.len() > 0

def nonempty_list(s):
    return s.apply(lambda v: is_seq(v) and len(v) > 0)

coverage = pd.Series({
    'has_metadata_record': items['_has_meta'].mean(),
    'has_title': nonempty_str(items.title).mean(),
    'has_store_brand': nonempty_str(items.store).mean(),
    'has_description': nonempty_list(items.description).mean(),
    'has_features': nonempty_list(items.features).mean(),
    'has_categories': nonempty_list(items.categories).mean(),
    'has_price': nonempty_str(items.price).mean(),
}).round(4)
display((coverage * 100).round(1).to_frame('% of items'))

missing_title_share = 1 - coverage['has_title']
inter_missing_title = items.loc[~nonempty_str(items.title), 'n_inter'].sum() / items.n_inter.sum()
print(f'Items without title: {missing_title_share:.1%} of items, {inter_missing_title:.1%} of interactions')

In [ ]:
def word_len(lst):
    return sum(len(str(x).split()) for x in lst) if is_seq(lst) else 0

items['title_words'] = items.title.fillna('').astype(str).str.split().str.len()
items['desc_words'] = items.description.apply(word_len)
items['feat_words'] = items.features.apply(word_len)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, col, name in zip(axes, ['title_words', 'desc_words', 'feat_words'], ['title', 'description', 'features']):
    vals = items[col].clip(upper=items[col].quantile(0.99))
    ax.hist(vals, bins=40, color=PRIMARY)
    ax.set_title(f'{name} length (words)'); ax.set_xlabel('words')
savefig('02_text_lengths')
items[['title_words', 'desc_words', 'feat_words']].describe(percentiles=[.5, .9, .99]).round(1)

## 6. Category tree
`categories` is a path such as `Beauty & Personal Care > Skin Care > Face > ...`. Level 1 (the second element)
is the natural sub-category for a coherent subset.

In [ ]:
def cat_level(lst, level):
    return lst[level] if is_seq(lst) and len(lst) > level else None

items['cat_l1'] = items.categories.apply(lambda c: cat_level(c, 1))
items['cat_l2'] = items.categories.apply(lambda c: cat_level(c, 2))
print('Level-0 values:', items.categories.apply(lambda c: cat_level(c, 0)).value_counts().head(5).to_dict())

cat_tbl = (items.groupby('cat_l1', dropna=False)
           .agg(items=('parent_asin', 'size'), interactions=('n_inter', 'sum'))
           .sort_values('interactions', ascending=False))
cat_tbl['interaction_share_pct'] = (100 * cat_tbl.interactions / cat_tbl.interactions.sum()).round(1)
display(cat_tbl.head(15))

top = cat_tbl.drop(index=[None, np.nan], errors='ignore').head(12).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh([str(x) for x in top.index], top.interactions, color=PRIMARY, height=0.6)
plt.title('Interactions by level-1 sub-category'); plt.xlabel('interactions')
savefig('02_category_l1')

## 7. Subset comparison
For each candidate we restrict the interactions, then **re-apply iterative 5-core** (removing items can push users
below 5 and vice versa, so we repeat until nothing changes). We then measure what each candidate would give us.

**Candidates**
- the largest level-1 sub-categories (a coherent catalog, e.g. "Skin Care")
- random samples of users (keeps the full catalog diversity)

**Decision rules (fixed before looking at the results)**
1. Users must be within `[min_users, max_users]`: enough clients for federated and DP experiments, small enough for
   affordable LLM teacher generation.
2. Items must be within `[min_items, max_items]`: large enough for a meaningful ranking and on-device compression
   study, small enough for full-catalog evaluation.
3. Among candidates that pass, prefer a **sub-category** (coherent catalog, sharper semantics for Model 1), then the
   one with the **longest average sequence**, then the most interactions.
4. If no sub-category passes, choose the user sample that passes with the most users.

In [ ]:
def kcore(df, k, user_col='user_id', item_col='parent_asin', verbose=False):
    it = 0
    while True:
        n = len(df)
        uc = df.groupby(user_col, observed=True)[user_col].transform('size')
        ic = df.groupby(item_col, observed=True)[item_col].transform('size')
        df = df[(uc >= k) & (ic >= k)]
        it += 1
        if verbose:
            print(f'  iteration {it}: {n:,} -> {len(df):,}')
        if len(df) == n:
            return df, it

K = CFG['kcore']
T = CFG['subset_targets']
base_df = ratings[['user_id', 'parent_asin', 'timestamp']].drop_duplicates(['user_id', 'parent_asin'])
item_cat = items.set_index('parent_asin')['cat_l1']
base_df = base_df.assign(cat_l1=base_df.parent_asin.map(item_cat))
# categorical dtype makes the repeated group-bys much faster on millions of rows
base_df['user_id'] = base_df['user_id'].astype('category')
base_df['parent_asin'] = base_df['parent_asin'].astype('category')

candidates = []
for cat in cat_tbl.drop(index=[None, np.nan], errors='ignore').head(CFG['subset_top_categories']).index:
    candidates.append(('category', cat))
for n in CFG['user_sample_sizes']:
    candidates.append(('user_sample', n))

rows = []
all_users = np.sort(ratings.user_id.unique().astype(str))
for kind, value in candidates:
    t0 = time.time()
    if kind == 'category':
        sub = base_df[base_df.cat_l1 == value]
    else:
        rng = np.random.default_rng(SEED)
        chosen = rng.choice(all_users, size=min(value, len(all_users)), replace=False)
        sub = base_df[base_df.user_id.isin(chosen)]
    sub_k, iters = kcore(sub, K)
    s = core_stats(sub_k) if len(sub_k) else {'users': 0, 'items': 0, 'interactions': 0}
    rows.append({'kind': kind, 'value': value, 'kcore_iterations': iters, **s,
                 'retained_pct_of_candidate': round(100 * len(sub_k) / max(len(sub), 1), 1),
                 'seconds': round(time.time() - t0, 1)})
    print(f'{kind:12s} {str(value):35s} -> users={s["users"]:>8,} items={s["items"]:>7,} inter={s["interactions"]:>9,}')

comp = pd.DataFrame(rows)
comp['passes_users'] = comp.users.between(T['min_users'], T['max_users'])
comp['passes_items'] = comp['items'].between(T['min_items'], T['max_items'])
comp['passes'] = comp.passes_users & comp.passes_items
comp.to_csv(REPORTS / 'subset_comparison.csv', index=False)
comp

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ok, bad = comp[comp.passes], comp[~comp.passes]
ax.scatter(bad['items'], bad.users, s=40, color=MUTED, label='outside targets')
ax.scatter(ok['items'], ok.users, s=50, color=PRIMARY, label='within targets')
for _, r in comp.iterrows():
    label = str(r.value)[:22] if r.kind == 'category' else f'{int(r.value):,}-user sample'
    ax.annotate(label, (r['items'], r.users), fontsize=7, xytext=(4, 3), textcoords='offset points')
ax.add_patch(plt.Rectangle((T['min_items'], T['min_users']), T['max_items'] - T['min_items'],
                           T['max_users'] - T['min_users'], fill=False, edgecolor=PRIMARY, linestyle='--', linewidth=1))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('items after 5-core'); ax.set_ylabel('users after 5-core')
ax.set_title('Candidate subsets (dashed box = target range)'); ax.legend(fontsize=8)
savefig('02_subset_comparison')

## 8. Recommendation
The rule below picks the subset automatically. To choose a different one, set `MANUAL_CHOICE`, for example
`{'kind': 'category', 'value': 'Skin Care'}` or `{'kind': 'user_sample', 'value': 100000}`, and re-run this cell.

In [ ]:
MANUAL_CHOICE = None

passing = comp[comp.passes]
if MANUAL_CHOICE is not None:
    choice_row = comp[(comp.kind == MANUAL_CHOICE['kind']) & (comp.value.astype(str) == str(MANUAL_CHOICE['value']))].iloc[0]
    reason = 'Manual choice by the team.'
elif (passing.kind == 'category').any():
    choice_row = passing[passing.kind == 'category'].sort_values(['mean_per_user', 'interactions'], ascending=False).iloc[0]
    reason = ('Sub-category within the user and item targets with the longest average sequence: a coherent catalog '
              'gives sharper item semantics for Model 1 while keeping enough users for Models 2 and 4.')
elif len(passing):
    choice_row = passing.sort_values('users', ascending=False).iloc[0]
    reason = 'No sub-category met the targets; the passing random user sample with the most users was chosen.'
else:
    comp['dist'] = (np.log(comp.users.clip(lower=1) / np.sqrt(T['min_users'] * T['max_users'])) ** 2 +
                    np.log(comp['items'].clip(lower=1) / np.sqrt(T['min_items'] * T['max_items'])) ** 2)
    choice_row = comp.sort_values('dist').iloc[0]
    reason = 'No candidate met all targets; the candidate closest to the centre of the target ranges was chosen. Review manually.'

subset_choice = {
    'kind': choice_row.kind,
    'value': choice_row.value if choice_row.kind == 'category' else int(choice_row.value),
    'seed': SEED,
    'expected_after_kcore': {k: int(choice_row[k]) for k in ['users', 'items', 'interactions']},
    'mean_seq_len': float(choice_row.mean_per_user),
    'reason': reason,
    'targets': T,
}
save_json(subset_choice, INTERIM / 'subset_choice.json')
print(json.dumps(subset_choice, indent=2))

## 9. Save EDA summary

In [ ]:
eda_summary = {
    'raw_stats': stats_raw,
    'quality': quality,
    'metadata_coverage': coverage.to_dict(),
    'items_without_title_pct': round(100 * missing_title_share, 2),
    'interactions_on_items_without_title_pct': round(100 * inter_missing_title, 2),
    'top20pct_items_interaction_share': round(float(top20_share), 4),
    'time_span': [str(ratings.dt.min()), str(ratings.dt.max())],
    'subset_choice': subset_choice,
}
save_json(eda_summary, REPORTS / 'eda_summary.json')
print('Saved', REPORTS / 'eda_summary.json')

## Findings to carry forward
| Finding | Decision it drives |
|---|---|
| Repeat user–item pairs (section 4) | `repeat_policy = keep_first` in notebook 03 |
| Timestamp ties within users (section 4) | tie-break by original file order (`row_id`) in notebook 05 |
| Items without title (section 5) | dropped in notebook 03 so all five models share one item set |
| Long tail (section 2) | head / mid / tail item groups in notebook 05 |
| Subset comparison (section 7) | subset applied in notebook 04 |

**Next:** `03_cleaning.ipynb`